In [ ]:
import cv2
import numpy as np

def order_points(pts):
    """Order 4 points: tl, tr, br, bl"""
    pts = np.array(pts, dtype="float32")
    s = pts.sum(axis=1)
    diff = np.diff(pts, axis=1)
    tl = pts[np.argmin(s)]
    br = pts[np.argmax(s)]
    tr = pts[np.argmin(diff)]
    bl = pts[np.argmax(diff)]
    return np.array([tl, tr, br, bl], dtype="float32")

def four_point_warp(img, quad):
    """Perspective warp using 4 corners"""
    quad = order_points(quad)
    (tl, tr, br, bl) = quad

    W = int(max(np.linalg.norm(tr - tl), np.linalg.norm(br - bl)))
    H = int(max(np.linalg.norm(bl - tl), np.linalg.norm(br - tr)))

    dst = np.array([[0, 0], [W - 1, 0], [W - 1, H - 1], [0, H - 1]], dtype="float32")
    M = cv2.getPerspectiveTransform(quad, dst)
    return cv2.warpPerspective(img, M, (W, H))

def warp_minarearect_from_black_regions(
    img_bgr,
    upscale=8,
    remove_blue_box=True
):
    # 1) Upscale cho dễ bắt biên
    up = cv2.resize(img_bgr, None, fx=upscale, fy=upscale, interpolation=cv2.INTER_CUBIC)

    # 2) (Optional) remove bbox màu xanh (nếu ảnh bạn có overlay)
    if remove_blue_box:
        hsv = cv2.cvtColor(up, cv2.COLOR_BGR2HSV)
        lower_blue = np.array([90, 80, 80])
        upper_blue = np.array([140, 255, 255])
        blue_mask = cv2.inRange(hsv, lower_blue, upper_blue)
        up = cv2.inpaint(up, blue_mask, 3, cv2.INPAINT_TELEA)

    # 3) Mask vùng tối (viền đen + chữ)
    hsv2 = cv2.cvtColor(up, cv2.COLOR_BGR2HSV)
    # V <= 110 (bạn có thể chỉnh 90-140 tuỳ ảnh tối/sáng)
    dark = cv2.inRange(hsv2, np.array([0, 0, 0]), np.array([180, 255, 110]))

    # 4) Morphology để nối liền các mảng tối
    dark = cv2.morphologyEx(
        dark, cv2.MORPH_CLOSE,
        cv2.getStructuringElement(cv2.MORPH_RECT, (9, 9)),
        iterations=2
    )

    # 5) Lấy “biên” của mảng tối để ra dạng khung tốt hơn
    grad = cv2.morphologyEx(
        dark, cv2.MORPH_GRADIENT,
        cv2.getStructuringElement(cv2.MORPH_RECT, (9, 9))
    )
    grad = cv2.dilate(
        grad, cv2.getStructuringElement(cv2.MORPH_RECT, (7, 7)),
        iterations=1
    )
    grad = cv2.morphologyEx(
        grad, cv2.MORPH_CLOSE,
        cv2.getStructuringElement(cv2.MORPH_RECT, (13, 13)),
        iterations=2
    )

    # 6) Contour lớn nhất → minAreaRect
    cnts, _ = cv2.findContours(grad, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        raise RuntimeError("Không tìm thấy contour. Thử tăng V-threshold hoặc chỉnh kernel.")

    c = max(cnts, key=cv2.contourArea)
    rect = cv2.minAreaRect(c)         # ((cx,cy),(w,h), angle)
    box = cv2.boxPoints(rect)         # 4 điểm
    box = box.astype(np.float32)

    # 7) Warp perspective
    warped = four_point_warp(up, box)

    return warped, up, grad, box

if __name__ == "__main__":
    # ====== Đổi path ảnh của bạn ở đây ======
    img_path = "plate_crop.png"
    img = cv2.imread(img_path)
    if img is None:
        raise FileNotFoundError(f"Không đọc được ảnh: {img_path}")

    warped, up_clean, mask_used, box = warp_minarearect_from_black_regions(
        img,
        upscale=8,
        remove_blue_box=True
    )

    # Lưu kết quả
    cv2.imwrite("plate_warped_minarearect.png", warped)

    # (Optional) vẽ quad để debug
    vis = up_clean.copy()
    cv2.polylines(vis, [order_points(box).astype(int)], True, (0, 255, 0), 3)
    cv2.imwrite("plate_quad_minarearect.png", vis)
    cv2.imwrite("plate_mask_used.png", mask_used)

    print("Saved:")
    print("- plate_warped_minarearect.png")
    print("- plate_quad_minarearect.png")
    print("- plate_mask_used.png")
